# Escalamiento de quejas financieras en Colombia
## Pipeline integral de Big Data con Machine Learning — Proyecto final

**Gerencia y almacenamiento en Big Data** · Autor: **Juan Rojas** · Modalidad individual

**Dataset:** *Quejas interpuestas por los consumidores financieros en contra de las entidades vigiladas en la plataforma Smartsupervision* — Superintendencia Financiera de Colombia (datos.gov.co, id `xyy7-rn7p`).

**Video de sustentación:** _[pendiente: pegar aquí el enlace no listado]_

**Repositorio:** https://github.com/ingjuancrojasl-crypto/Rojas_Juan_BigData_ProyectoFinal

| Sección | Fase CRISP-DM | Estado |
|---|---|---|
| 1. Business Understanding y selección del dataset | Business Understanding | ✅ |
| 2. Decisión de arquitectura: pandas o Spark | — | ✅ |
| 3. ETL (exploración, calidad, 4 técnicas, variables derivadas) | Data Understanding + Data Preparation | ✅ |
| 4. Análisis exploratorio y visualizaciones | Data Understanding | ⏳ |
| 5. Machine Learning | Modeling + Evaluation | ⏳ |
| 6. Insights en MongoDB | Deployment | ⏳ |
| 7. Reporte PDF | Deployment | ⏳ |

In [1]:
# Librerías y configuración común
import os
import sys
import json
import time
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import psutil

# El notebook vive en notebooks/; el código reutilizable en src/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import config
import etl

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Raíz del proyecto:", RAIZ)
print("pandas", pd.__version__, "| numpy", np.__version__, "| Python", platform.python_version())

Raíz del proyecto: /home/claude/Rojas_Juan_BigData_ProyectoFinal
pandas 3.0.5 | numpy 2.5.3 | Python 3.13.18


---
# 1. Business Understanding y selección del dataset

## 1.1 Contexto
Smartsupervision es la plataforma con la que la **Superintendencia Financiera de Colombia (SFC)** recibe y almacena las quejas de los consumidores financieros contra las entidades que vigila (bancos, aseguradoras, fondos de pensiones, fiduciarias, comisionistas, SEDPE, etc.). Cada registro del dataset es un **conteo mensual** de quejas para una combinación de entidad, instancia de recepción, motivo, producto y municipio.

Una queja puede atenderse en la **propia entidad vigilada** o **escalar** a una instancia externa: el **Defensor del Consumidor Financiero**, la **propia SFC** o una remisión por competencia. El escalamiento es costoso para todos: la entidad pierde la oportunidad de resolver, el consumidor tarda más en obtener respuesta y el supervisor asume carga operativa.

## 1.2 Descripción del dataset

In [2]:
# Descarga (si no existe) y metadatos del archivo crudo
if not config.ARCHIVO_RAW.exists():
    import subprocess
    subprocess.run([sys.executable, str(RAIZ / "src" / "descargar_datos.py")], check=True)

metadata_descarga = json.loads(config.ARCHIVO_METADATA_DESCARGA.read_text(encoding="utf-8"))
df_crudo = etl.cargar_crudo()

descripcion = pd.Series({
    "Fuente": f"{config.DATASET_ENTIDAD} — datos.gov.co",
    "Página del dataset": config.URL_PAGINA_DATASET,
    "URL de descarga usada": metadata_descarga["url_descarga"],
    "Fecha de descarga": metadata_descarga["fecha_descarga"],
    "Periodo cubierto": f"{df_crudo['Año_Creacion'].min()}-{df_crudo['Mes_Creacion'][df_crudo['Año_Creacion'] == df_crudo['Año_Creacion'].min()].min():02d} a "
                        f"{df_crudo['Año_Creacion'].max()}-{df_crudo['Mes_Creacion'][df_crudo['Año_Creacion'] == df_crudo['Año_Creacion'].max()].max():02d}",
    "Filas": f"{len(df_crudo):,}",
    "Columnas": df_crudo.shape[1],
    "Total de quejas representadas": f"{df_crudo['Cantidad_quejas_recibidas'].sum():,}",
    "Peso en disco (CSV)": f"{config.ARCHIVO_RAW.stat().st_size / 1e6:,.1f} MB",
    "Peso en memoria (pandas por defecto)": f"{etl.memoria_mb(df_crudo):,.1f} MB",
}, name="valor").to_frame()
descripcion

,valor
Fuente,Superintendencia Financiera de Colombia (SFC) ...
Página del dataset,https://www.datos.gov.co/d/xyy7-rn7p
URL de descarga usada,https://raw.githubusercontent.com/ingjuancroja...
Fecha de descarga,2026-11-12T00:29:16
Periodo cubierto,2023-01 a 2026-09
Filas,"1,400,425"
Columnas,10
Total de quejas representadas,"10,041,272"
Peso en disco (CSV),221.8 MB
Peso en memoria (pandas por defecto),279.1 MB


**¿Por qué este dataset?**
- Es **real, público y oficial** (SFC vía datos.gov.co) y se actualiza mensualmente.
- Supera ampliamente el volumen pedido: **1,4 millones de registros** que representan **más de 10 millones de quejas**.
- No viene limpio: tiene nulos, catálogos de texto inconsistentes, llaves ambiguas, códigos sin descripción y meses faltantes (ver sección 3), lo que permite aplicar el ETL completo.
- Tiene una **variable objetivo natural** para clasificación (la instancia de recepción) y varias dimensiones para análisis OLAP (tiempo × geografía × producto × entidad).
- Responde a un problema con impacto real en la protección al consumidor financiero.

## 1.3 Pregunta de negocio
> **¿Qué características de una queja financiera (producto, motivo, tipo de entidad, ubicación y momento) aumentan la probabilidad de que escale a una instancia externa —Defensor del Consumidor o Superintendencia— en lugar de resolverse en la propia entidad vigilada?**

## 1.4 Preguntas específicas
1. ¿Cómo evolucionó el volumen de quejas entre 2023 y 2026 y qué sectores explican el crecimiento?
2. ¿Qué productos y categorías de motivo concentran las quejas y cuáles tienen la **mayor tasa de escalamiento**?
3. ¿Qué regiones y departamentos presentan tasas de escalamiento más altas que el promedio nacional?
4. ¿Qué tipos de entidad resuelven mejor internamente sus quejas y cuáles las dejan escalar más?
5. ¿Qué variables pesan más en la predicción del escalamiento y qué acciones preventivas sugieren?

## 1.5 Tarea de Machine Learning y criterio de éxito
- **Tarea:** clasificación binaria. Variable objetivo `escalada` = 1 si la instancia de recepción es distinta de *Entidad vigilada* (Defensor, SFC o remisión), 0 en caso contrario.
- **Unidad de análisis:** cada registro (combinación mensual de entidad × instancia × motivo × producto × municipio).
- **Validación:** división **temporal** — entrenamiento con 2023-2025 y prueba con 2026 (sin meses incompletos), para simular el uso real del modelo sobre quejas futuras.
- **Línea base:** clasificador sencillo (clase mayoritaria / estratificado y una regla por tasa histórica del producto).
- **Criterio de éxito:** que el mejor modelo **supere el F1 de la clase "escalada" de la mejor línea base en al menos 10 puntos porcentuales**, sin perder interpretabilidad para el negocio.

---
# 2. Decisión de arquitectura: pandas o Spark

Retomando los criterios de la Actividad 4, la decisión se toma con **mediciones propias**: tamaño en disco, memoria, recursos de la máquina y tiempos de lectura y procesamiento.

## 2.1 Recursos de la máquina

In [3]:
memoria_total_gb = psutil.virtual_memory().total / 1e9
recursos = pd.Series({
    "Sistema operativo": f"{platform.system()} {platform.release()}",
    "Núcleos de CPU (lógicos)": psutil.cpu_count(logical=True),
    "Memoria RAM total": f"{memoria_total_gb:,.1f} GB",
    "Memoria RAM disponible": f"{psutil.virtual_memory().available / 1e9:,.1f} GB",
    "¿Dentro de Docker?": Path("/.dockerenv").exists(),
}, name="valor").to_frame()
recursos

,valor
Sistema operativo,Linux 6.18.44-fc-v378
Núcleos de CPU (lógicos),2
Memoria RAM total,8.4 GB
Memoria RAM disponible,7.3 GB
¿Dentro de Docker?,False


## 2.2 Mediciones: lectura por defecto, tipos optimizados, lectura por bloques y Parquet

In [4]:
mediciones = []

def medir(nombre, funcion):
    inicio = time.perf_counter()
    resultado = funcion()
    segundos = time.perf_counter() - inicio
    return resultado, segundos

# a) Lectura con tipos por defecto
df_defecto, t_defecto = medir("defecto", lambda: etl.cargar_crudo(optimizado=False))
mediciones.append(("CSV, tipos por defecto", t_defecto, etl.memoria_mb(df_defecto)))

# b) Lectura con tipos optimizados (category + enteros pequeños)
df_optimizado, t_optimizado = medir("optimizado", lambda: etl.cargar_crudo(optimizado=True))
mediciones.append(("CSV, tipos optimizados", t_optimizado, etl.memoria_mb(df_optimizado)))

# c) Lectura por bloques: total de quejas por año sin cargar todo el archivo
def total_por_bloques(tamano_bloque=200_000):
    parciales = []
    for bloque in pd.read_csv(config.ARCHIVO_RAW, encoding="utf-8-sig", chunksize=tamano_bloque,
                              usecols=["Año_Creacion", "Cantidad_quejas_recibidas"]):
        parciales.append(bloque.groupby("Año_Creacion")["Cantidad_quejas_recibidas"].sum())
    return pd.concat(parciales).groupby(level=0).sum()

total_bloques, t_bloques = medir("bloques", total_por_bloques)
mediciones.append(("CSV por bloques de 200.000 filas (2 columnas)", t_bloques, np.nan))

# d) Parquet: escritura y lectura del mismo DataFrame optimizado
ruta_parquet_prueba = config.DIR_PROCESSED / "_prueba_lectura.parquet"
_, t_escritura = medir("parquet_w", lambda: df_optimizado.to_parquet(ruta_parquet_prueba, index=False))
df_parquet, t_parquet = medir("parquet_r", lambda: pd.read_parquet(ruta_parquet_prueba))
mediciones.append(("Parquet (lectura)", t_parquet, etl.memoria_mb(df_parquet)))

tabla_mediciones = pd.DataFrame(mediciones, columns=["escenario", "segundos", "memoria_mb"])
tabla_mediciones["segundos"] = tabla_mediciones["segundos"].round(2)

tamanos = pd.Series({
    "CSV crudo (MB)": round(config.ARCHIVO_RAW.stat().st_size / 1e6, 1),
    "Parquet snappy (MB)": round(ruta_parquet_prueba.stat().st_size / 1e6, 1),
    "Reducción de memoria con tipos optimizados": f"{1 - etl.memoria_mb(df_optimizado) / etl.memoria_mb(df_defecto):.1%}",
    "Memoria del dataset / RAM total": f"{etl.memoria_mb(df_defecto) / (memoria_total_gb * 1000):.1%}",
}, name="valor")

ruta_parquet_prueba.unlink()
del df_defecto, df_parquet
display(tabla_mediciones)
display(tamanos.to_frame())
print("Verificación lectura por bloques (quejas por año):", total_bloques.to_dict())

,escenario,segundos,memoria_mb
0,"CSV, tipos por defecto",2.01,279.10
1,"CSV, tipos optimizados",1.61,25.30
2,CSV por bloques de 200.000 filas (2 columnas),0.87,NaN
3,Parquet (lectura),0.10,25.30


,valor
CSV crudo (MB),221.80
Parquet snappy (MB),9.00
Reducción de memoria con tipos optimizados,90.9%
Memoria del dataset / RAM total,3.3%


Verificación lectura por bloques (quejas por año): {2023: 1953978, 2024: 3070007, 2025: 3178925, 2026: 1838362}


## 2.3 Decisión: **pandas**

| Criterio (Actividad 4) | Medición propia | Implicación |
|---|---|---|
| Volumen | ~1,4 M filas × 10 columnas; ~220 MB en CSV | Mediano: cabe holgadamente en una sola máquina |
| Memoria | ~280 MB con tipos por defecto → **~25 MB** con tipos optimizados (−91 %) | ~3 % de la RAM de la máquina; con tipos optimizados, una fracción mínima |
| Tiempo | Lectura completa en ~2 s; Parquet en < 0,1 s; ETL completo en < 30 s | No hay cuello de botella que justifique distribuir |
| Infraestructura | Un solo nodo (portátil o contenedor) | Spark añadiría JVM, configuración de SparkSession y sobrecarga de planificación sin beneficio |
| Crecimiento | El dataset crece ~30 MB/año | Seguiría siendo manejable con pandas por varios años |

**Conclusión:** se usa **pandas** con tres técnicas para manejar el volumen:
1. **Tipos optimizados** (`category` para textos repetidos, enteros de 8/16/32 bits): reduce la memoria ~91 %.
2. **Lectura por bloques** (`chunksize`) disponible para agregaciones sobre el CSV sin cargarlo completo, útil si el dataset creciera.
3. **Formato Parquet** (columnar y comprimido): 222 MB de CSV → ~9 MB de Parquet, y se lee ~20 veces más rápido. El dashboard lee el Parquet, nunca el CSV.

Spark sería la opción correcta si el volumen superara la memoria de una máquina (decenas de GB) o si se integraran otras fuentes masivas (por ejemplo, quejas a nivel individual de varios años).

---
# 3. ETL con CRISP-DM

## 3.1 Data Understanding: exploración inicial (antes de limpiar)
### Estructura y primeras filas

In [5]:
print(f"Dimensiones: {df_crudo.shape[0]:,} filas × {df_crudo.shape[1]} columnas")
df_crudo.head()

Dimensiones: 1,400,425 filas × 10 columnas


,Año_Creacion,Mes_Creacion,Tipo_Entidad,Codigo_Entidad,Instancia_Recepcion,Motivo,Producto,Departamento,Municipio,Cantidad_quejas_recibidas
0,2026,9,1,2,Entidad vigilada,Negación injustificada a la apertura del producto,Cuenta de ahorro,"BOGOTÁ, D.C.","BOGOTÁ, D.C.",2
1,2026,9,13,44,Entidad vigilada,Cobros a póliza terminada,Seguro de desempleo,QUINDÍO,ARMENIA,1
2,2026,9,5,12,Otra (remisión por competencia),Reconocimiento de prestaciones económicas - In...,Fiducia de administración,ATLÁNTICO,BARRANQUILLA,1
3,2026,9,5,12,Entidad vigilada,Reconocimiento de prestaciones económicas - In...,Pensiones obligatorias - Vejez,CHOCÓ,QUIBDÓ,1
4,2026,9,13,26,Entidad vigilada,Error en el pago de la indemnización o suma as...,Seguro de automóviles,"BOGOTÁ, D.C.","BOGOTÁ, D.C.",1


### Tipos, nulos y valores únicos por columna

In [6]:
etl.perfil_columnas(df_crudo)

,tipo,nulos,pct_nulos,valores_unicos
Año_Creacion,int64,0,0.00,4
Mes_Creacion,int64,0,0.00,12
Tipo_Entidad,int64,0,0.00,23
Codigo_Entidad,int64,0,0.00,99
Instancia_Recepcion,str,0,0.00,4
Motivo,str,0,0.00,316
Producto,str,0,0.00,90
Departamento,str,5157,0.37,33
Municipio,str,5157,0.37,1036
Cantidad_quejas_recibidas,int64,0,0.00,1790


### Estadísticas descriptivas de las variables numéricas

In [7]:
df_crudo.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).T

,count,mean,std,min,25%,50%,75%,90%,99%,max
Año_Creacion,"1,400,425.00","2,024.39",1.06,"2,023.00","2,023.00","2,024.00","2,025.00","2,026.00","2,026.00","2,026.00"
Mes_Creacion,"1,400,425.00",6.23,3.32,1.00,3.00,6.00,9.00,11.00,12.00,12.00
Tipo_Entidad,"1,400,425.00",8.89,17.63,1.00,1.00,1.00,13.00,22.00,128.00,401.00
Codigo_Entidad,"1,400,425.00",25.87,24.85,1.00,7.00,14.00,42.00,54.00,124.00,157.00
Cantidad_quejas_recibidas,"1,400,425.00",7.17,160.58,1.00,1.00,1.00,2.00,5.00,69.00,"37,366.00"


La variable `Cantidad_quejas_recibidas` está **muy sesgada**: la mediana es 1 y el percentil 99 es menor a 100, pero el máximo supera las 37.000 quejas en un solo registro. No son errores: son combinaciones de alto volumen (grandes bancos, motivos masivos como *transacción no reconocida*).

### Distribución de las variables categóricas principales

In [8]:
for columna in ["Instancia_Recepcion", "Tipo_Entidad", "Producto"]:
    resumen = (df_crudo.groupby(columna)["Cantidad_quejas_recibidas"]
               .agg(registros="size", quejas="sum")
               .sort_values("quejas", ascending=False).head(8))
    resumen["pct_quejas"] = (resumen["quejas"] / df_crudo["Cantidad_quejas_recibidas"].sum() * 100).round(1)
    display(resumen)

,registros,quejas,pct_quejas
Instancia_Recepcion,,,
Entidad vigilada,1013998,8812304,87.80
Superintendencia Financiera de Colombia,245388,641625,6.40
Defensor del consumidor financiero,96806,496629,4.90
Otra (remisión por competencia),44233,90714,0.90


,registros,quejas,pct_quejas
Tipo_Entidad,,,
1,801444,8513837,84.80
14,167582,407393,4.10
13,164941,355743,3.50
25,54606,150733,1.50
23,45185,131809,1.30
4,55008,124748,1.20
22,33721,117717,1.20
128,13992,92955,0.90


,registros,quejas,pct_quejas
Producto,,,
Cuenta de ahorro,226450,2999760,29.90
Depósitos de bajo monto (Capítulo I del Título 15 del Decreto 2555 de 2010),172021,2727438,27.20
Tarjetas de crédito,165975,1662128,16.60
Creditos de consumo,95455,443861,4.40
Otros productos de Establecimientos de Crédito y SEDPE,66864,344148,3.40
Créditos de vivienda,46767,213872,2.10
Pensiones obligatorias - Vejez,69639,210046,2.10
Seguro de riesgos laborales,73270,204765,2.00


### Cobertura temporal: quejas por año y mes

In [9]:
cobertura = df_crudo.pivot_table(index="Mes_Creacion", columns="Año_Creacion",
                                 values="Cantidad_quejas_recibidas", aggfunc="sum")
cobertura

Año_Creacion,2023,2024,2025,2026
Mes_Creacion,,,,
1,"147,541.00","196,723.00","249,979.00","248,495.00"
2,"144,690.00","216,596.00","252,137.00",NaN
3,"152,889.00","215,622.00","264,479.00","290,743.00"
4,"137,270.00","253,835.00","260,168.00","256,876.00"
5,"152,981.00","256,740.00","291,973.00","257,985.00"
6,"153,347.00","243,429.00","251,461.00","246,860.00"
7,"162,950.00","279,168.00","281,647.00","277,923.00"
8,"170,175.00","287,183.00","246,359.00","253,578.00"
9,"163,515.00","283,735.00","264,604.00","5,902.00"


La tabla revela dos problemas que no se ven en el conteo de nulos: **febrero de 2026 no existe** en el dataset y **septiembre de 2026 está cargado parcialmente** (unas 6.000 quejas frente a ~250.000 de un mes normal).

## 3.2 Tabla de problemas de calidad detectados
Cada fila se calcula sobre los datos crudos (no está escrita a mano) y se asocia a la técnica ETL que la resuelve.

In [10]:
problemas_calidad = etl.diagnosticar_calidad(df_crudo)
problemas_calidad

,categoria,problema,registros_afectados,tratamiento
0,Nombres de columna,"Mayúsculas, tildes y 'ñ' en encabezados; archi...",10,Limpieza: renombrar a snake_case y leer con ut...
1,Nulos,Registros sin departamento ni municipio,5157,Limpieza: imputar 'SIN INFORMACIÓN' (no se eli...
2,Inconsistencia de texto,Motivos repetidos que solo cambian en mayúscul...,7,Normalización: clave sin tildes y etiqueta can...
3,Catálogo mixto,Filas con motivos de un catálogo distinto escr...,1490,Normalización: unificar a la etiqueta canónica
4,Llave ambigua,Códigos de entidad que se repiten en varios ti...,49,Enriquecimiento: llave compuesta id_entidad = ...
5,Llave ambigua,Nombres de municipio que existen en más de un ...,67,Enriquecimiento: llave compuesta departamento|...
6,Códigos sin descripción,Tipos de entidad solo como número,23,Enriquecimiento: cruce con catálogo de tipos d...
7,Distribución sesgada,"Cantidad de quejas > p99 (69); máximo 37,366",13849,Normalización: log1p y bandera de volumen atíp...
8,Meses faltantes,Meses sin ningún registro: 2026-02,1,Análisis: no interpolar; comparar años con pro...
9,Meses con carga parcial,Meses con < 20 % del volumen mediano: 2026-09,2467,Limpieza: bandera mes_incompleto; se excluyen ...


## 3.3 Data Preparation: las 4 técnicas ETL
Cada paso devuelve un registro con conteos **antes / después** para dejar evidencia del efecto.

### Técnica 1 — Limpieza
- Renombrar encabezados a `snake_case` sin tildes ni `ñ` (y leer con `utf-8-sig` para quitar el BOM).
- Imputar ubicación faltante con `SIN INFORMACIÓN` y crear la bandera `sin_ubicacion`. **No se eliminan** esos registros porque una proporción alta corresponde a quejas recibidas directamente por la SFC: eliminarlos sesgaría la variable objetivo.
- Validar rangos (mes 1-12, año plausible, cantidad > 0) y eliminar duplicados exactos.
- Marcar con `mes_incompleto` los meses de carga parcial.

In [11]:
df_limpio, registro_limpieza = etl.limpiar(df_crudo)
pd.DataFrame(registro_limpieza)

,etapa,tecnica,antes,despues,diferencia,detalle
0,Renombrar columnas,Limpieza,10,10,0,Encabezados a snake_case sin tildes ni 'ñ'
1,Imputar ubicación faltante,Limpieza,5157,0,-5157,Nulos -> 'SIN INFORMACIÓN' + bandera sin_ubica...
2,Eliminar valores inválidos,Limpieza,1400425,1400425,0,"Mes en 1-12, año plausible y cantidad > 0"
3,Eliminar duplicados exactos,Limpieza,1400425,1400425,0,Filas idénticas en todas las columnas
4,Marcar meses con carga parcial,Limpieza,0,2467,2467,Filas en meses incompletos (2026-09); se conse...
5,Total limpieza (filas),Limpieza,1400425,1400425,0,


In [12]:
# Evidencia: los registros sin ubicación escalan mucho más que el promedio
sin_ubicacion = df_limpio.groupby("sin_ubicacion")["instancia_recepcion"].value_counts(normalize=True).unstack().round(3)
sin_ubicacion.index = ["Con ubicación", "Sin ubicación"]
sin_ubicacion

instancia_recepcion,Defensor del consumidor financiero,Entidad vigilada,Otra (remisión por competencia),Superintendencia Financiera de Colombia
Con ubicación,0.07,0.72,0.03,0.17
Sin ubicación,0.02,0.53,0.01,0.44


### Técnica 2 — Normalización
- **Textos:** se construye una clave sin tildes, en mayúsculas y sin puntuación final; las variantes con la misma clave se unifican en la escritura con más quejas. Se agregan dos sinónimos revisados a mano (`OTROS` → `Otros motivos`, `portafolios` → `portafolio`). En municipios la unificación se hace **dentro de cada departamento** para no fusionar homónimos (ej. *Chimá*, Córdoba, y *Chima*, Santander).
- **Ortografía:** `Creditos` → `Créditos` en productos.
- **Escala:** `log_cantidad_quejas = log(1 + cantidad)` y bandera `volumen_atipico` (> p99). Los atípicos se conservan porque son volúmenes reales.
- **Tipos:** `category` para textos y enteros pequeños.

In [13]:
df_normalizado, registro_normalizacion = etl.normalizar(df_limpio)
pd.DataFrame(registro_normalizacion)

,etapa,tecnica,antes,despues,diferencia,detalle
0,Unificar escritura de motivo,Normalización,316,307,-9,"Valores únicos antes/después (tildes, mayúscul..."
1,Unificar escritura de producto,Normalización,90,90,0,"Valores únicos antes/después (tildes, mayúscul..."
2,Unificar escritura de municipio,Normalización,1037,1037,0,"Valores únicos antes/después (tildes, mayúscul..."
3,Unificar escritura de instancia_recepcion,Normalización,4,4,0,"Valores únicos antes/después (tildes, mayúscul..."
4,Corregir tildes en producto,Normalización,95455,0,-95455,Filas con 'Creditos' sin tilde antes/después
5,Escala log y bandera de atípicos,Normalización,0,13849,13849,log1p(cantidad); atípico si cantidad > p99 = 69
6,Optimizar tipos (MB en memoria),Normalización,282,35,-247,category para textos y enteros de 8/16/32 bits


In [14]:
# Evidencia: ejemplos de motivos unificados
variantes = (df_limpio.assign(clave=df_limpio["motivo"].map(etl.clave_texto).replace(etl.SINONIMOS_MOTIVO))
             .groupby("clave")["motivo"].unique())
variantes = variantes[variantes.map(len) > 1].map(lambda v: " | ".join(v))
variantes.to_frame("variantes encontradas")

,variantes encontradas
clave,
DISTRIBUCION DE PORTAFOLIO,Distribución de portafolio | Distribución de p...
ERROR EN LA LIQUIDACION O PAGO ERRONEO,Error en la liquidación o pago erróneo | ERROR...
"INCONSISTENCIAS EN LA APLICACION DE APORTES, RETIROS Y CANCELACIONES","Inconsistencias en la aplicación de aportes, r..."
INDEBIDA CLASIFICACION DEL PERFIL DE RIESGO,Indebida clasificación del perfil de riesgo | ...
NEGACION INJUSTIFICADA A LA APERTURA DEL PRODUCTO,Negación injustificada a la apertura del produ...
OTROS MOTIVOS,Otros motivos | OTROS
PUBLICIDAD ENGANOSA,Publicidad engañosa | PUBLICIDAD ENGAÑOSA
REPORTE DE PAGO DESACTUALIZADO,Reporte de pago desactualizado | REPORTE DE PA...
RETIRO COMPLETO DEL DEPOSITO,Retiro completo del depósito | RETIRO COMPLETO...


### Técnica 3 — Agregación
1. **Consolidación:** después de normalizar, algunas filas describen exactamente la misma combinación de las 9 dimensiones; se suman sus quejas. Se verifica que el **total de quejas no cambie**.
2. **Cubos OLAP:** agregaciones multidimensionales que se usarán como filtros del dashboard (se construyen después del enriquecimiento porque usan región, sector y categoría de motivo).

In [15]:
df_consolidado, registro_agregacion = etl.consolidar(df_normalizado)
pd.DataFrame(registro_agregacion)

,etapa,tecnica,antes,despues,diferencia,detalle
0,Consolidar registros equivalentes,Agregación,1400425,1400348,-77,Filas con las mismas 9 dimensiones tras normal...
1,Control: total de quejas,Agregación,10041272,10041272,0,Debe ser igual: la consolidación no pierde quejas


### Técnica 4 — Enriquecimiento
- **Catálogo de tipos de entidad SFC** (`data/reference/tipos_entidad_sfc.csv`): nombre y sector de cada código. La columna `fuente_nombre` indica si el nombre proviene del catálogo publicado por la SFC o si se infirió a partir de los productos que reporta ese tipo de entidad.
- **Regiones naturales y código DANE** (`data/reference/regiones_departamentos.csv`).
- **Llaves compuestas:** `id_entidad = tipo-código` (el código solo no es único) e `id_municipio = departamento | municipio`.
- **Variables derivadas:** `periodo` (fecha), `trimestre`, `categoria_motivo` (agrupa 300+ motivos en 10 categorías de negocio con reglas de palabras clave) y la variable objetivo `escalada`.

In [16]:
df_final, registro_enriquecimiento = etl.enriquecer(df_consolidado)
pd.DataFrame(registro_enriquecimiento)

,etapa,tecnica,antes,despues,diferencia,detalle
0,Cruce con catálogo de tipos de entidad,Enriquecimiento,1400348,1400348,0,Filas con nombre y sector de entidad
1,Cruce con regiones naturales (DANE),Enriquecimiento,1400348,1395191,-5157,Filas con región asignada (el resto no tiene u...
2,Columnas agregadas,Enriquecimiento,14,25,11,"nombre/sector de entidad, región, DANE, llaves..."


In [17]:
# Evidencia: catálogo cruzado y categorías de motivo
display(df_final.groupby(["sector", "nombre_tipo_entidad", "fuente_nombre"], observed=True)["cantidad_quejas"]
        .sum().sort_values(ascending=False).head(10).to_frame("quejas"))
display(df_final.groupby("categoria_motivo", observed=True)["cantidad_quejas"].sum()
        .sort_values(ascending=False).to_frame("quejas"))

quejas
sector                      nombre_tipo_entidad                                fuente_nombre                  
Bancario y crédito          Establecimientos bancarios                         Catálogo SFC            8513837
Seguros                     Compañías de seguros de vida                       Inferido por productos   407393
                            Compañías de seguros generales                     Inferido por productos   355743
Pensiones y cesantías       Régimen de prima media (pensiones públicas)        Inferido por productos   150733
                            Sociedades administradoras de fondos de pension... Catálogo SFC             131809
Bancario y crédito          Compañías de financiamiento                        Catálogo SFC             124748
                            Instituciones oficiales especiales                 Catálogo SFC             117717
Pagos digitales             Entidades de depósitos de bajo monto y pagos       Inferido por productos    92955
Fiduciario                  Sociedades fiduciarias                             Catálogo SFC              67775
Otros servicios financieros Operadores de información PILA                     Inferido por productos    41450

,quejas
categoria_motivo,
Fraude y transacciones no reconocidas,2898900
Errores en pagos y transacciones,2334948
"Crédito, cobranza y centrales de riesgo",872589
Gestión de productos y vinculación,822471
Canales y calidad del servicio,806170
"Cobros, tarifas y tasas",693505
"Información, asesoría y datos",502654
Pensiones y cesantías,479785
Seguros y siniestros,363184


## 3.4 Variables derivadas para el modelo
| Variable | Tipo | Cómo se construye | Aporte esperado al modelo |
|---|---|---|---|
| `escalada` | Objetivo (0/1) | Instancia ≠ *Entidad vigilada* | Variable a predecir |
| `categoria_motivo` | Categórica (10) | Reglas de palabras clave sobre el motivo | Reduce 307 motivos a grupos interpretables |
| `sector` / `nombre_tipo_entidad` | Categórica | Cruce con catálogo SFC | Diferencias de gestión entre industrias |
| `region` | Categórica (6 + sin dato) | Cruce con regiones DANE | Patrones geográficos de escalamiento |
| `trimestre` | Numérica (1-4) | A partir del mes | Estacionalidad |
| `log_cantidad_quejas` | Numérica | `log(1 + cantidad)` | Volumen de la combinación sin el sesgo extremo |
| `volumen_atipico` | Binaria | cantidad > p99 | Identifica combinaciones masivas |
| `sin_ubicacion` | Binaria | Departamento nulo en origen | Fuertemente asociada a quejas recibidas por la SFC |

La tasa de escalamiento por variable derivada confirma que aportan señal:

In [18]:
def tasa_escalamiento(df, columna):
    tabla = (df.assign(quejas_escaladas=df["cantidad_quejas"] * df["escalada"])
             .groupby(columna, observed=True)
             .agg(quejas=("cantidad_quejas", "sum"), escaladas=("quejas_escaladas", "sum"),
                  registros=("escalada", "size"), registros_escalados=("escalada", "sum")))
    tabla["tasa_quejas"] = (tabla["escaladas"] / tabla["quejas"]).round(3)
    tabla["tasa_registros"] = (tabla["registros_escalados"] / tabla["registros"]).round(3)
    return tabla.sort_values("tasa_registros", ascending=False)

display(tasa_escalamiento(df_final, "categoria_motivo"))
display(tasa_escalamiento(df_final, "sector"))

,quejas,escaladas,registros,registros_escalados,tasa_quejas,tasa_registros
categoria_motivo,,,,,,
"Crédito, cobranza y centrales de riesgo",872589,331683,161363,73505,0.38,0.46
Otros motivos,267066,91921,70875,29484,0.34,0.42
Seguros y siniestros,363184,108913,152823,51148,0.30,0.34
Pensiones y cesantías,479785,85689,143058,45261,0.18,0.32
"Información, asesoría y datos",502654,83946,110892,34541,0.17,0.31
"Cobros, tarifas y tasas",693505,60485,80861,20894,0.09,0.26
Fraude y transacciones no reconocidas,2898900,180198,152253,33807,0.06,0.22
Gestión de productos y vinculación,822471,107972,177464,36470,0.13,0.21
Canales y calidad del servicio,806170,68263,137224,27195,0.09,0.20


,quejas,escaladas,registros,registros_escalados,tasa_quejas,tasa_registros
sector,,,,,,
Fiduciario,67775,35129,32191,24037,0.52,0.75
Pensiones y cesantías,282542,67201,99782,30188,0.24,0.30
Mercado de valores,8228,1288,3890,1117,0.16,0.29
Bancario y crédito,8769143,991815,898692,252229,0.11,0.28
Pagos digitales,94490,11301,15034,4049,0.12,0.27
Seguros,776378,120997,342662,73960,0.16,0.22
Otros servicios financieros,42716,1237,8097,775,0.03,0.10


Hay dos formas de medir el escalamiento y cuentan historias distintas:
- **Por registro** (`tasa_registros`): proporción de combinaciones que escalan. Es la unidad que usará el modelo.
- **Por queja** (`tasa_quejas`): proporción de quejas que escalan. Es menor porque las combinaciones escaladas suelen tener pocas quejas cada una.

## 3.5 Cubos OLAP y almacenamiento en Parquet

In [19]:
cubos = etl.construir_cubos_olap(df_final)
for nombre, cubo in cubos.items():
    print(f"{nombre}: {len(cubo):,} filas")
cubos["cubo_anio_sector_instancia_categoria"].head(10)

cubo_anio_region_departamento_producto: 7,019 filas
cubo_anio_sector_instancia_categoria: 935 filas
serie_mensual_sector: 301 filas
ranking_entidades: 216 filas


,anio,sector,instancia_recepcion,categoria_motivo,registros,total_quejas,quejas_escaladas,tasa_escalamiento
0,2025,Bancario y crédito,Entidad vigilada,Fraude y transacciones no reconocidas,32924,940050,0,0.00
1,2024,Bancario y crédito,Entidad vigilada,Errores en pagos y transacciones,42993,878812,0,0.00
2,2024,Bancario y crédito,Entidad vigilada,Fraude y transacciones no reconocidas,32048,787542,0,0.00
3,2025,Bancario y crédito,Entidad vigilada,Errores en pagos y transacciones,54302,706441,0,0.00
4,2026,Bancario y crédito,Entidad vigilada,Fraude y transacciones no reconocidas,19646,538696,0,0.00
5,2023,Bancario y crédito,Entidad vigilada,Fraude y transacciones no reconocidas,26458,430255,0,0.00
6,2026,Bancario y crédito,Entidad vigilada,Errores en pagos y transacciones,29938,323108,0,0.00
7,2023,Bancario y crédito,Entidad vigilada,Canales y calidad del servicio,21132,250632,0,0.00
8,2023,Bancario y crédito,Entidad vigilada,Errores en pagos y transacciones,23633,212303,0,0.00
9,2025,Bancario y crédito,Entidad vigilada,Gestión de productos y vinculación,30486,183839,0,0.00


In [20]:
# Ejemplo de operaciones OLAP sobre el cubo año × región × departamento × producto
cubo_geo = cubos["cubo_anio_region_departamento_producto"]

# Slice: solo 2025
slice_2025 = cubo_geo[cubo_geo["anio"] == 2025]
# Dice: 2025, región Caribe, productos de tarjeta o cuenta de ahorro
dice = slice_2025[(slice_2025["region"] == "Caribe") &
                  (slice_2025["producto"].isin(["Tarjetas de crédito", "Cuenta de ahorro"]))]
# Drill-down: región -> departamento
drill_region = slice_2025.groupby("region", observed=True)[["total_quejas", "quejas_escaladas"]].sum()
drill_departamento = (slice_2025[slice_2025["region"] == "Caribe"]
                      .groupby("departamento", observed=True)[["total_quejas", "quejas_escaladas"]].sum()
                      .sort_values("total_quejas", ascending=False))
display(dice.head())
display(drill_region.sort_values("total_quejas", ascending=False))
display(drill_departamento.head())

,anio,region,departamento,producto,registros,total_quejas,quejas_escaladas,tasa_escalamiento
70,2025,Caribe,ATLÁNTICO,Cuenta de ahorro,3433,16281,1415,0.09
123,2025,Caribe,ATLÁNTICO,Tarjetas de crédito,2109,9338,1101,0.12
139,2025,Caribe,BOLÍVAR,Cuenta de ahorro,2363,8566,652,0.08
227,2025,Caribe,BOLÍVAR,Tarjetas de crédito,1200,4881,517,0.11
236,2025,Caribe,MAGDALENA,Cuenta de ahorro,1706,4551,353,0.08


,total_quejas,quejas_escaladas
region,,
Andina,2706571,312334
Caribe,232890,16847
Pacífica,175130,17358
Orinoquía,40211,3545
Amazonía,16017,1071
SIN INFORMACIÓN,6470,1057
Insular,1636,111


,total_quejas,quejas_escaladas
departamento,,
ATLÁNTICO,89990,6591
BOLÍVAR,48376,3184
MAGDALENA,25341,2059
CESAR,23265,1961
CÓRDOBA,22569,1558


In [21]:
# Guardar el dataset limpio y las agregaciones en data/processed/
resumen_etl = {
    "filas_crudas": len(df_crudo), "columnas_crudas": df_crudo.shape[1],
    "filas_finales": len(df_final), "columnas_finales": len(etl.COLUMNAS_SALIDA),
    "registros_eliminados": len(df_crudo) - len(df_limpio),
    "registros_consolidados": len(df_normalizado) - len(df_consolidado),
    "total_quejas": int(df_final["cantidad_quejas"].sum()),
    "problemas_detectados": problemas_calidad.to_dict(orient="records"),
    "pasos_etl": registro_limpieza + registro_normalizacion + registro_agregacion + registro_enriquecimiento,
    "agregaciones": {n: len(c) for n, c in cubos.items()},
}
etl.guardar_resultados(df_final, cubos, resumen_etl)

df_verificacion = pd.read_parquet(config.ARCHIVO_PARQUET_LIMPIO)
print(f"Parquet limpio: {config.ARCHIVO_PARQUET_LIMPIO.stat().st_size / 1e6:,.1f} MB en disco, "
      f"{df_verificacion.shape[0]:,} filas × {df_verificacion.shape[1]} columnas, "
      f"{etl.memoria_mb(df_verificacion):,.1f} MB en memoria")
print("Archivos de agregación:", sorted(p.name for p in config.DIR_AGREGACIONES.glob("*.parquet")))

Parquet limpio: 9.5 MB en disco, 1,400,348 filas × 25 columnas, 63.1 MB en memoria
Archivos de agregación: ['cubo_anio_region_departamento_producto.parquet', 'cubo_anio_sector_instancia_categoria.parquet', 'ranking_entidades.parquet', 'serie_mensual_sector.parquet']


### Resumen antes / después del ETL completo

In [22]:
pd.DataFrame({
    "antes (crudo)": [f"{len(df_crudo):,}", df_crudo.shape[1], f"{etl.memoria_mb(df_crudo):,.1f} MB",
                      int(df_crudo.isna().sum().sum()), df_crudo["Motivo"].nunique(),
                      f"{df_crudo['Cantidad_quejas_recibidas'].sum():,}"],
    "después (limpio)": [f"{len(df_verificacion):,}", df_verificacion.shape[1],
                         f"{etl.memoria_mb(df_verificacion):,.1f} MB",
                         int(df_verificacion.drop(columns=["codigo_dane_departamento"]).isna().sum().sum()),
                         df_verificacion["motivo"].nunique(),
                         f"{df_verificacion['cantidad_quejas'].sum():,}"],
}, index=["Filas", "Columnas", "Memoria", "Valores nulos", "Motivos distintos", "Total de quejas"])

,antes (crudo),después (limpio)
Filas,"1,400,425","1,400,348"
Columnas,10,25
Memoria,279.1 MB,63.1 MB
Valores nulos,10314,0
Motivos distintos,316,307
Total de quejas,"10,041,272","10,041,272"


**Nota:** las únicas celdas vacías que quedan están en `codigo_dane_departamento` para los registros *SIN INFORMACIÓN*, que no tienen departamento al cual asignar código; se dejan vacías a propósito para no inventar un código.

---
## Próximos pasos
- **Sección 4:** análisis exploratorio con al menos 6 visualizaciones (principios de la Actividad 7).
- **Sección 5:** modelo de clasificación de escalamiento: línea base, 2+ modelos, métricas, interpretación y guardado en `models/`.
- **Secciones 6-7:** documento de análisis en MongoDB, consultas y reporte PDF automático.
- Dashboard Streamlit y despliegue completo con `docker compose up --build`.